# Phase 4 (CJPE) — c-6 encoder-decoder: LORMA + KD
Run the cells top to bottom. Set every `None` in E2 and E3 and write `CJPEHead` before running them.

## Cell 0a — imports + HF login

In [38]:
# --- Phase 1 c-6 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## Cell 0b — tokenizer (Phase 1)

In [39]:
# --- Phase 1 c-6 notebook, cell 8 (verbatim): SPECIAL_TOKENS, BPETokenizer, train_tokenizer ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok


## Cell 0c — model classes (Phase 1)

In [40]:
# --- Phase 1 c-6 notebook, cells 20, 22 and 23 (verbatim, in this order) ---
# Cell 23 must come after cell 22: it redefines CausalSelfAttention (q/k/v_proj), which the
# saved decoder weights use. Without it, Cell A fails with qkv_proj key errors in decoder_layers.

# ---- cell 20 ----
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---- cell 22 ----
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Seq2SeqConfig:
    vocab_size: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    n_layers: int = 8

# ---------------------------------------------------------
# 1. Attention Modules
# ---------------------------------------------------------
class EncoderSelfAttention(nn.Module):
    """Bidirectional self-attention for the encoder (no causal mask)."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CausalSelfAttention(nn.Module):
    """Causal self-attention for the decoder (looks only at past tokens)."""
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CrossAttention(nn.Module):
    """Cross-attention: Query from Decoder, Key/Value from Encoder."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        B_c, T_c, C_c = context.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(context).chunk(2, dim=-1)
        k, v = [t.view(B_c, T_c, self.n_heads, self.d_head).transpose(1, 2) for t in kv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

# ---------------------------------------------------------
# 2. Feed-Forward & Standard Blocks
# ---------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


class EncoderBlock(nn.Module):
    """Standard bidirectional Transformer encoder block."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = EncoderSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Standard causal Transformer decoder block with cross-attention."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout)
        
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), encoder_out)
        x = x + self.ffn(self.ln3(x))
        return x


# ---------------------------------------------------------
# 4. Hourglass Blocks (Encoder & Decoder versions)
# ---------------------------------------------------------
class EncoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([EncoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor):
        residual = x
        shortened_x = self.downsample(x) # No shift for Encoder
        for block in self.bottleneck:
            shortened_x = block(shortened_x)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual


class DecoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([DecoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor):
        residual = x
        x_shifted = shift_right(x, shift=self.factor - 1) # Shift required for Decoder causality
        shortened_x = self.downsample(x_shifted)
        for block in self.bottleneck:
            shortened_x = block(shortened_x, encoder_out)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual

# ---------------------------------------------------------
# 5. Top-Level Hourglass Seq2Seq Model
# ---------------------------------------------------------
class StandardEncoderDecoder(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)

        self.encoder_layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.encoder_ln = nn.LayerNorm(cfg.d_model)

        self.decoder_layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.decoder_ln = nn.LayerNorm(cfg.d_model)

        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def get_positional_embeddings(self, seq_len, device):
        return self.pos_emb(torch.arange(seq_len, device=device).unsqueeze(0))

    def encode(self, enc_idx):
        x = self.tok_emb(enc_idx) + self.get_positional_embeddings(enc_idx.size(1), enc_idx.device)
        for block in self.encoder_layers:
            x = block(x)
        return self.encoder_ln(x)

    def decode(self, dec_idx, encoder_out):
        x = self.tok_emb(dec_idx) + self.get_positional_embeddings(dec_idx.size(1), dec_idx.device)
        for block in self.decoder_layers:
            x = block(x, encoder_out)
        return self.lm_head(self.decoder_ln(x))

    def forward(self, enc_idx, dec_idx):
        return self.decode(dec_idx, self.encode(enc_idx))


# ---- cell 23 ----
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 2000
    max_seq_len: int = 256
    dropout: float = 0.15
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


@dataclass
class EncDecConfig(Seq2SeqConfig):
    enc_len: int = None      # tokens the encoder reads (prompt side)
    dec_len: int = None      # tokens the decoder learns to produce (continuation side)


class EncDecLM(nn.Module):
    def __init__(self, cfg: EncDecConfig):
        super().__init__()
        assert None not in (cfg.enc_len, cfg.dec_len), "set enc_len and dec_len in the config"
        self.cfg = cfg
        self.core = StandardEncoderDecoder(cfg)

    def encode(self, enc_idx):
        return self.core.encode(enc_idx)

    def decode(self, dec_idx, encoder_out):
        return self.core.decode(dec_idx, encoder_out)

    def forward(self, enc_idx, dec_idx):
        return self.core(enc_idx, dec_idx)


## Cell A — load c-6 + tokenizer

In [41]:
# =====================================================================================
# CELL A — load the Phase 1 c-6 encoder-decoder and its tokenizer
# Expect <All keys matched successfully>, 55.13M params, vocab 11022, d_model 560, 8 layers, max_seq_len 1536
# =====================================================================================
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudraalt/phase-1-c-6-simple-encoder-decoder"   # TODO: the folder the find command showed (up to, not including, cs781_phase1_full)
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/model_encdec/bpe_tokenizer.pkl", "rb"))   # c-6's own tokenizer
ck = torch.load(f"{ROOT}/cs781_phase1_full/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
cfg = EncDecConfig(**ck["config"])
lm = EncDecLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| n_layers", cfg.n_layers, "| max_seq_len", cfg.max_seq_len)


<All keys matched successfully>
params 55.13008 M | vocab 11022 | d_model 560 | n_layers 8 | max_seq_len 1536


## Cell B — encoder features check

In [42]:
# =====================================================================================
# CELL B — per-token vectors from the c-6 ENCODER + sanity check
# The encoder is bidirectional: every token sees the tokens on both sides. The decoder is not used.
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor, T <= 1536
    c = lm.core
    T = idx.size(1)
    assert T <= c.cfg.max_seq_len         # learned position table has 1536 rows
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_layers:          # 8 bidirectional EncoderBlocks
        x = blk(x)
    return c.encoder_ln(x)                # (B, T, 560)

# Sanity check: same as the model's own encoder
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f, ref = features(idx), lm.encode(idx)
print("max diff", (f - ref).abs().max().item(), tuple(f.shape))   # expect 0.0, (1, 64, 560)


max diff 0.0 (1, 64, 560)


## Cell C — CJPE data

In [43]:
# =====================================================================================
# CELL C — Phase 4 (CJPE) data: load train / dev / test cases and check the label field
# One case = one long judgment -> one binary label (0 / 1).
# =====================================================================================
import json
from collections import Counter
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
print("Phase 4 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_4" in f])

def get4(name):
    """Reads Phase_4_CJPE/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"Phase_4_CJPE/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

_raw = get4("Train/train.json")
_first = _raw[0] if isinstance(_raw, list) else next(iter(_raw.values()))
print("train layout:", type(_raw).__name__, len(_raw), "| first record keys:",
      list(_first) if isinstance(_first, dict) else type(_first).__name__)

TEXT_FIELDS  = ("text", "document", "case_text", "judgment", "sentences")
LABEL_FIELDS = ("label", "labels", "outcome", "decision", "target", "y", "pred")

def as_cases(raw, name):
    """-> {case_id: {"text": str, "label": 0/1 or None}}, in file order.
    Accepts a list of dicts, {case_id: {...}}, a one-key wrapper like {"data": [...]},
    or columns {"id": [...], "text": [...], ("label": [...])}. Other fields (expert_1..5) are ignored.
    Repeated ids (train merges the single- and multi-instance sets):
      same text and label as the first copy -> exact duplicate, dropped;
      different text or label              -> kept as a separate case "<id>#dup<n>" (train/dev),
                                              or dropped with a warning (test: one prediction per id)."""
    while isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict)):
        raw = next(iter(raw.values()))
    if isinstance(raw, dict) and "id" in raw and isinstance(raw["id"], list):
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw["id"]))]
    items = [(d.get("id", n), d) for n, d in enumerate(raw)] if isinstance(raw, list) else list(raw.items())
    cases, n_exact, n_diff, n_conflict = {}, 0, 0, 0
    for k, d in items:
        assert isinstance(d, dict), f"case {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"case {k}: no text field (keys {list(d)})"
        text = d[tf]
        if isinstance(text, list):                    # a list of sentences/paragraphs
            text = " ".join(map(str, text))
        lf = next((f for f in LABEL_FIELDS if f in d), None)
        lab = d[lf] if lf is not None else None
        if isinstance(lab, list):
            assert len(lab) == 1, f"case {k}: label is a list {lab}"
            lab = lab[0]
        rec = {"text": str(text), "label": int(lab) if lab is not None else None}
        if k in cases:
            first = cases[k]
            if first["text"] == rec["text"] and first["label"] == rec["label"]:
                n_exact += 1
                continue
            if name == "test":
                print(f"WARNING test id {k} appears twice with different text: keeping the first")
                continue
            n_diff += 1
            n_conflict += first["text"] == rec["text"] and first["label"] != rec["label"]
            j = 1
            while f"{k}#dup{j}" in cases:
                j += 1
            k = f"{k}#dup{j}"
        cases[k] = rec
    print(f"{name}: {len(items)} records -> {len(cases)} cases | exact duplicates dropped {n_exact} | "
          f"repeated ids with different text/label kept {n_diff} (same text, different label: {n_conflict})")
    return cases

train = as_cases(_raw, "train")
dev   = as_cases(get4("Train/dev.json"), "dev")
test  = as_cases(get4("Test/test.json"), "test")
del _raw

for name, split in (("train", train), ("dev", dev), ("test", test)):
    labs = Counter(d["label"] for d in split.values())
    chars = sorted(len(d["text"]) for d in split.values())
    print(f"{name:5s}: {len(split):6d} cases | labels {dict(labs)} | chars median {chars[len(chars)//2]} max {chars[-1]}")
assert all(d["label"] in (0, 1) for s in (train, dev) for d in s.values()), "train/dev labels must be 0/1: check LABEL_FIELDS"
# The test file carries a label field too (placeholder or hidden): never use it, it is cleared here
_tl = Counter(d["label"] for d in test.values())
if any(v is not None for v in _tl):
    print(f"test label field values (ignored, set to None): {dict(_tl)}")
for d in test.values():
    d["label"] = None
print("first test ids:", list(test)[:3])
print("note: write_prediction uses the test ids above, so every test id gets exactly one entry")
print(f"train∩dev ids {len(set(train) & set(dev))} | train∩test {len(set(train) & set(test))}")


Phase 4 files on HF: ['Phase_4_CJPE/Test/test.json', 'Phase_4_CJPE/Train/dev.json', 'Phase_4_CJPE/Train/train.json']
train layout: list 37387 | first record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
train: 37387 records -> 32305 cases | exact duplicates dropped 5082 | repeated ids with different text/label kept 0 (same text, different label: 0)
dev: 1988 records -> 994 cases | exact duplicates dropped 994 | repeated ids with different text/label kept 0 (same text, different label: 0)
test: 1517 records -> 1517 cases | exact duplicates dropped 0 | repeated ids with different text/label kept 0 (same text, different label: 0)
train:  32305 cases | labels {0: 18920, 1: 13385} | chars median 12120 max 2492701
dev  :    994 cases | labels {0: 497, 1: 497} | chars median 16708 max 260198
test :   1517 cases | labels {1: 762, 0: 755} | chars median 17320 max 233000
test label field values (ignored, set to None): {1: 762, 0: 755}
first test ids: [

## Cell D — tokenise, scorer, writer, baseline

In [44]:
# =====================================================================================
# CELL D — document tokenisation, local scorer, prediction writer, majority baseline
# =====================================================================================
import json, os, zipfile
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1
CLASSES = (0, 1)

def encode_doc(text):
    ids = tok.encode(text)
    return ids if ids else [UNK_ID]

# Tokenise every case once: d["ids"] = token ids of the whole judgment (re-run if you change encode_doc)
for name, split in (("train", train), ("dev", dev), ("test", test)):
    for d in tqdm(split.values(), desc=f"tokenising {name}"):
        d["ids"] = encode_doc(d["text"])

# Length statistics in c-6 tokens, to choose CHUNK_LEN (<= 1536), STRIDE and MAX_CHUNKS in Cell E
for name, split in (("train", train), ("dev", dev), ("test", test)):
    L = np.array([len(d["ids"]) for d in split.values()])
    print(f"{name:5s} tokens: min {L.min()} median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
          f"p99 {np.percentile(L, 99):.0f} max {L.max()} | > 1536: {(L > 1536).mean():.1%}")
for lab in CLASSES:
    L = np.array([len(d["ids"]) for d in train.values() if d["label"] == lab])
    print(f"train label {lab}: {len(L)} cases, median tokens {np.median(L):.0f}")


# -------------------------------------------------------------------------------------
# Local scorer: macro-F1 / macro-P / macro-R over the two classes (= sklearn average="macro")
# -------------------------------------------------------------------------------------
def binary_scores(gold, pred):
    gold, pred = np.asarray(gold), np.asarray(pred)
    per = {}
    for c in CLASSES:
        tp = int(((pred == c) & (gold == c)).sum())
        fp = int(((pred == c) & (gold != c)).sum())
        fn = int(((pred != c) & (gold == c)).sum())
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        per[c] = (P, R, 2 * P * R / (P + R) if P + R else 0.0, tp + fn)        # last = gold support
    m = lambda i: sum(v[i] for v in per.values()) / len(CLASSES)
    return {"p": m(0), "r": m(1), "f1": m(2), "acc": float((gold == pred).mean()), "per": per}

def evaluate(pred, docs, show=True):
    """pred: {case_id: 0/1}. Scores every case in docs."""
    ids = list(docs)
    res = binary_scores([docs[k]["label"] for k in ids], [pred[k] for k in ids])
    if show:
        print(f"macro-F1 {res['f1']:.4f}  P {res['p']:.4f}  R {res['r']:.4f}  acc {res['acc']:.4f} | " +
              " | ".join(f"class {c}: F1 {v[2]:.3f} (n={v[3]})" for c, v in res["per"].items()))
    return res

def confusion(pred, docs):
    """Rows = gold, columns = predicted."""
    M = np.zeros((2, 2), dtype=int)
    for k, d in docs.items():
        M[d["label"], pred[k]] += 1
    print(f"confusion (rows gold, cols pred)\n        pred 0  pred 1\ngold 0 {M[0,0]:7d} {M[0,1]:7d}\ngold 1 {M[1,0]:7d} {M[1,1]:7d}")
    return M

try:   # cross-check against sklearn once
    from sklearn.metrics import f1_score
    _g, _p = [0, 1, 1, 0, 1], [0, 1, 0, 0, 0]
    assert abs(binary_scores(_g, _p)["f1"] - f1_score(_g, _p, average="macro")) < 1e-9
    print("scorer matches sklearn macro-F1")
except ImportError:
    pass


# -------------------------------------------------------------------------------------
# Writer: prediction.json = LIST of {"id", "pred"} with integer pred, in test order, zipped
# -------------------------------------------------------------------------------------
def write_prediction(pred, test_docs, out_dir="/kaggle/working"):
    os.makedirs(out_dir, exist_ok=True)
    out = []
    for case_id in test_docs:
        p = int(pred[case_id])                       # numpy ints / bools -> plain int
        assert p in (0, 1), f"{case_id}: pred {p!r} is not 0/1"
        out.append({"id": case_id, "pred": p})
    assert len(out) == len(test_docs) and len({o["id"] for o in out}) == len(out), "one entry per test case"
    assert set(pred) >= set(test_docs), "missing test ids"
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    back = json.load(open(path))
    assert isinstance(back, list) and all(type(o["pred"]) is int and set(o) == {"id", "pred"} for o in back)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
    print(f"{len(out)} cases | predicted label counts {dict(Counter(o['pred'] for o in out))} -> {zpath}")


# -------------------------------------------------------------------------------------
# Majority baseline: checks the scorer and writer end to end before any modelling
# -------------------------------------------------------------------------------------
majority = Counter(d["label"] for d in train.values()).most_common(1)[0][0]
print("majority label:", majority)
evaluate({k: majority for k in dev}, dev)
write_prediction({k: majority for k in test}, test, out_dir="/kaggle/working/baseline")   # format check only; don't spend a slot on it


tokenising train:   0%|          | 0/32305 [00:00<?, ?it/s]

tokenising dev:   0%|          | 0/994 [00:00<?, ?it/s]

tokenising test:   0%|          | 0/1517 [00:00<?, ?it/s]

train tokens: min 55 median 2692 p90 8584 p99 25630 max 541168 | > 1536: 69.1%
dev   tokens: min 217 median 4234 p90 10414 p99 30573 max 70685 | > 1536: 93.0%
test  tokens: min 107 median 4416 p90 10912 p99 27295 max 64341 | > 1536: 92.3%
train label 0: 18920 cases, median tokens 2668
train label 1: 13385 cases, median tokens 2724
scorer matches sklearn macro-F1
majority label: 0
macro-F1 0.3333  P 0.2500  R 0.5000  acc 0.5000 | class 0: F1 0.667 (n=497) | class 1: F1 0.000 (n=497)
1517 cases | predicted label counts {0: 1517} -> /kaggle/working/baseline/submission.zip


## Cell E0 — shared plumbing (Phase 2 windows)

In [45]:
# =====================================================================================
# CELL E0 — shared plumbing for the c-6 Phase 4 (CJPE) experiments, Phase 2 style
# Run once after Cells 0, A, B, C, D. Then run a technique cell: E2 (LORMA), then E3 (KD).
# Needs from earlier cells: lm, cfg, device, features, train, dev, test (with d["ids"]),
#   CLASSES, evaluate, confusion, write_prediction
#
# Windows exactly as in Phase 2: full-length windows of WINDOW tokens every STRIDE tokens,
# the last one ending at the document end; a short case is one window of its own length.
# The encoder has no padding mask, so windows are never padded: batches hold windows of
# IDENTICAL length. Every window of a case is trained with that case's label, and at
# prediction time the case's window probabilities are combined into one 0/1 decision.
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import math, time, random
from collections import defaultdict, Counter
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

N_CLASSES = 2
FEAT_DIM = cfg.d_model   # 560 for c-6

# The decoder is never used for classification; dropping it frees GPU memory. tok_emb stays (shared with lm_head).
lm.core.decoder_layers = None
lm.core.decoder_ln = None
torch.cuda.empty_cache()


# -------------------------------------------------------------------------------------
# c-6 encoder path (identical to Cell B features(), but on any backbone object)
# -------------------------------------------------------------------------------------
def encoder_forward(backbone, idx):
    c = backbone.core
    assert idx.size(1) <= c.cfg.max_seq_len, "window longer than 1536"
    x = c.tok_emb(idx) + c.get_positional_embeddings(idx.size(1), idx.device)
    for blk in c.encoder_layers:
        x = blk(x)
    return c.encoder_ln(x)                      # (B, T, 560)


def encoder_linears(backbone, which):
    """Yield (parent, attr, nn.Linear) for the chosen linears in every encoder block.
    which ⊆ {"qkv_proj", "out_proj", "ffn_in", "ffn_out"}.
    c-6 encoder attention has ONE fused qkv_proj (560 -> 1680), not q/k/v_proj."""
    for blk in backbone.core.encoder_layers:
        for w in which:
            if w in ("qkv_proj", "out_proj"):
                parent, attr = blk.self_attn, w
            elif w == "ffn_in":
                parent, attr = blk.ffn.net, "0"     # 560 -> d_ff
            elif w == "ffn_out":
                parent, attr = blk.ffn.net, "2"     # d_ff -> 560
            else:
                raise ValueError(w)
            yield parent, attr, getattr(parent, attr)


def wrap_linears(backbone, which, make_wrapper):
    """Replace each chosen linear by make_wrapper(linear). Returns how many were wrapped."""
    todo = list(encoder_linears(backbone, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)


class CJPEModel(nn.Module):
    """backbone (c-6 encoder path) -> head -> 2 logits per WINDOW.
    head: YOUR module, (B, T, 560) token vectors of B windows -> (B, 2)."""
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone, self.head = backbone, head
    def forward(self, idx):
        return self.head(encoder_forward(self.backbone, idx))   # (B, 2)


def identity_probe():
    """64 real tokens from a training case, for the 'adapter = identity at init' check."""
    d = next(iter(train.values()))
    return torch.tensor([d["ids"][:64]], device=device)


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Windows (Phase 2 make_windows, unchanged) + which windows stand for the case
#   DOC_REPR "all" = every window (exactly Phase 2). Judgments are long, so you can also keep
#   at most max_windows of them: "first", "last", "first_last" (half from each end) or
#   "spread" (evenly over the whole case). Windows stay in document order.
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]


def select_windows(wins, doc_repr, max_windows):
    n = len(wins)
    if doc_repr == "all" or n <= max_windows:
        return wins
    if doc_repr == "first":
        return wins[:max_windows]
    if doc_repr == "last":
        return wins[-max_windows:]
    if doc_repr == "first_last":
        h = (max_windows + 1) // 2
        return wins[:h] + wins[n - (max_windows - h):] if max_windows > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_windows - 1)) for i in range(max_windows)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)


def build_examples(docs, window, stride, doc_repr="all", max_windows=None):
    """One example per window: {"doc", "w", "ids", "label"} (label None for test)."""
    assert window <= cfg.max_seq_len and doc_repr in DOC_REPRS
    assert doc_repr == "all" or max_windows, "set max_windows for this doc_repr"
    examples = []
    for doc_id, d in tqdm(docs.items(), desc="windowing", leave=False):
        ids = d["ids"]
        wins = select_windows(make_windows(len(ids), window, stride), doc_repr, max_windows)
        for w, (s, e) in enumerate(wins):
            examples.append({"doc": doc_id, "w": w, "ids": ids[s:e], "label": d["label"]})
    per = Counter(ex["doc"] for ex in examples)
    print(f"{len(per)} cases -> {len(examples)} windows | windows per case: mean {len(examples)/len(per):.2f} max {max(per.values())}")
    return examples


def make_batches(examples, batch_size, shuffle):
    """Batches of windows with IDENTICAL length (no padding needed)."""
    by_len = defaultdict(list)
    for ex in examples:
        by_len[len(ex["ids"])].append(ex)
    out = []
    for exs in by_len.values():
        if shuffle:
            random.shuffle(exs)
        out += [exs[i:i + batch_size] for i in range(0, len(exs), batch_size)]
    if shuffle:
        random.shuffle(out)
    return out


def to_tensors(batch):
    b = {"ids": torch.tensor([ex["ids"] for ex in batch], device=device)}
    if batch[0]["label"] is not None:
        b["labels"] = torch.tensor([ex["label"] for ex in batch], device=device)
    if "soft" in batch[0]:
        b["soft"] = torch.stack([ex["soft"] for ex in batch]).to(device).float()   # (B, 2) teacher logits
    return b


def ce_loss(logits, labels, class_weights=None):
    """Plain window-level cross-entropy on the case's gold label."""
    return F.cross_entropy(logits.float(), labels, weight=class_weights)


def make_class_weights(scheme):
    """'inverse' (1/count), 'sqrt_inverse' (1/sqrt(count)) or 'none', from the train case counts,
    normalised to mean 1 so the loss scale stays similar."""
    assert scheme in ("inverse", "sqrt_inverse", "none"), "set WEIGHT_SCHEME"
    cnt = Counter(d["label"] for d in train.values())
    counts = torch.tensor([cnt.get(c, 0) for c in CLASSES], dtype=torch.float)
    w = None
    if scheme != "none":
        w = 1.0 / counts if scheme == "inverse" else 1.0 / counts.sqrt()
        w = w / w.mean()
    for c, n in zip(CLASSES, counts.tolist()):
        print(f"  class {c}: train n={int(n)}  weight={1.0 if w is None else w[c].item():.3f}")
    return None if w is None else w.to(device)


# -------------------------------------------------------------------------------------
# Prediction: window probabilities -> one P(label 1) per case -> 0/1
#   agg (yours): "mean" = average of the windows' P(1), "max" = largest P(1),
#                "vote" = share of windows whose P(1) >= 0.5
#   threshold (yours): case is 1 when the aggregated value >= threshold
# -------------------------------------------------------------------------------------
AGGS = ("mean", "max", "vote")

@torch.no_grad()
def predict_probs(model, examples, batch_size, agg):
    assert agg in AGGS, agg
    model.eval()
    per_doc = defaultdict(list)
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        with torch.autocast("cuda", dtype=torch.float16):
            p1 = model(to_tensors(batch)["ids"]).float().softmax(-1)[:, 1].tolist()
        for ex, p in zip(batch, p1):
            per_doc[ex["doc"]].append(p)
    if agg == "mean":
        return {d: sum(v) / len(v) for d, v in per_doc.items()}
    if agg == "max":
        return {d: max(v) for d, v in per_doc.items()}
    return {d: sum(p >= 0.5 for p in v) / len(v) for d, v in per_doc.items()}


def predict(model, examples, batch_size, agg, threshold):
    return {d: int(p >= threshold) for d, p in predict_probs(model, examples, batch_size, agg).items()}


# -------------------------------------------------------------------------------------
# Training loop (Phase 2 train(), renamed train_model because `train` is the data from Cell C).
# loss_fn(logits, batch_tensors) comes from the technique cell, so KD can use the teacher's
# logits and LORMA plain CE. Keeps the epoch with the best dev macro-F1 (case level).
# lr_lambda: optional step -> multiplier (your schedule), or None for constant LR.
# -------------------------------------------------------------------------------------
def train_model(model, train_ex, val_ex, val_docs, loss_fn, *, lr, epochs, batch_size, agg, threshold,
                weight_decay=0.0, grad_clip=1.0, lr_lambda=None, save_path="best_trainable.pt"):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_res, log = time.time(), -1.0, None, []
    skipped = 0   # batches whose loss was NaN/inf (skipped, not trained on)
    for epoch in range(epochs):
        model.train(); tot, nb = 0.0, 0
        pbar = tqdm(make_batches(train_ex, batch_size, shuffle=True), desc=f"epoch {epoch + 1}/{epochs}")
        for batch in pbar:
            b = to_tensors(batch)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["ids"])
            loss = loss_fn(logits.float(), b)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched: sched.step()
            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{tot / nb:.4f}", skipped=skipped)
        res = evaluate(predict(model, val_ex, batch_size, agg, threshold), val_docs, show=False)
        log.append({"epoch": epoch + 1, "train_loss": tot / max(1, nb), "dev_f1": res["f1"],
                    "dev_p": res["p"], "dev_r": res["r"], "dev_acc": res["acc"], "skipped": skipped})
        print(log[-1])
        if res["f1"] > best_f1:
            best_f1, best_res = res["f1"], res
            torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, save_path)
    print(f"best dev macro-F1 {best_f1:.4f} (P {best_res['p']:.4f}, R {best_res['r']:.4f}) | "
          f"train time {(time.time() - t0) / 60:.1f} min | peak GPU mem {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")
    return log, best_res


@torch.no_grad()
def time_inference(model, examples, batch_size, agg):
    """Wall-clock seconds to predict a set (for the efficiency table in the report)."""
    torch.cuda.synchronize(); t = time.time()
    predict_probs(model, examples, batch_size, agg)
    torch.cuda.synchronize(); return time.time() - t


def error_cases(pred, probs, docs, k=5, chars=300):
    """A few false positives (gold 0, pred 1) and false negatives (gold 1, pred 0), most confident first."""
    for name, g, p in (("FALSE POSITIVES", 0, 1), ("FALSE NEGATIVES", 1, 0)):
        ids = [i for i, d in docs.items() if d["label"] == g and pred[i] == p]
        ids.sort(key=lambda i: -abs(probs[i] - 0.5))
        print(f"\n{name}: {len(ids)}")
        for i in ids[:k]:
            t = docs[i]["text"]
            print(f"- {i} | score {probs[i]:.3f} | {len(docs[i]['ids'])} tokens\n  start: {t[:chars]!r}\n  end:   {t[-chars:]!r}")


def write_submission(model, build_kwargs, batch_size, agg, threshold, best_path=None, out_dir="/kaggle/working"):
    """Reload the best trainable weights, predict test, write prediction.json + submission.zip (Cell D writer)."""
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex = build_examples(test, **build_kwargs)
    write_prediction(predict(model, test_ex, batch_size, agg, threshold), test, out_dir=out_dir)


## Cell E2 — LORMA

In [46]:
# =====================================================================================
# CELL E2 — LORMA on the c-6 encoder for CJPE (compulsory technique), Phase 2 style
# Run after Cells 0, A, B, C, D and E0 (cellE0_common_c6.py). To run again from scratch,
# re-run Cell A and E0 first (LORMA wraps the encoder's linears in place).
# Course rule: every value marked None and the head are yours.
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters — all yours. None = not set; the assert stops you.
# -------------------------------------------------------------------------------------
WINDOW        = 1024
STRIDE        = 512
DOC_REPR      = "all"
MAX_WINDOWS   = None

LORMA_RANK    = 200
LORMA_TARGETS = ("qkv_proj", "out_proj", "ffn_in", "ffn_out")   # subset of ("qkv_proj", "out_proj", "ffn_in", "ffn_out") (Phase 3 c-6: qkv_proj, out_proj)
LR            = 1e-4
EPOCHS        = 10
BATCH_SIZE    = 16
WEIGHT_DECAY  = 0.01
LR_LAMBDA     = None

WEIGHT_SCHEME = "sqrt_inverse"

AGG           = "mean"
THRESHOLD     = 0.5

SEED          = 42
for _n in ["WINDOW", "STRIDE", "DOC_REPR", "LORMA_RANK", "LORMA_TARGETS", "LR", "EPOCHS", "BATCH_SIZE",
           "WEIGHT_DECAY", "WEIGHT_SCHEME", "AGG", "THRESHOLD"]:
    assert globals()[_n] is not None, f"set {_n}"
assert STRIDE <= WINDOW <= cfg.max_seq_len
assert DOC_REPR == "all" or MAX_WINDOWS, "set MAX_WINDOWS"
BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS)
random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c6_lorma_{DOC_REPR}"
OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
SAVE = f"{OUT_DIR}/best_trainable.pt"

CLASS_WEIGHTS = make_class_weights(WEIGHT_SCHEME)


# -------------------------------------------------------------------------------------
# [L] LORMA layer (identity variant) — Rudra's code from Phase 3: W' = (I + BA) W
# -------------------------------------------------------------------------------------
class LORMALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int):
        super().__init__()
        self.base = base

        # Freeze pretrained weights
        for p in self.base.parameters():
            p.requires_grad = False

        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank

        # A: (rank, out_features), since A operates on Wx
        # B: (out_features, rank)
        self.lorma_A = nn.Parameter(
            torch.empty(self.rank, self.out_features)
        )
        self.lorma_B = nn.Parameter(
            torch.empty(self.out_features, self.rank)
        )

        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B)

    def forward(self, x):
        # Compute pretrained linear output without bias: Wx
        wx = F.linear(x, self.base.weight, bias=None)

        # Low-rank transformation of Wx: BA(Wx)
        lorma_out = F.linear(
            F.linear(wx, self.lorma_A),
            self.lorma_B
        )

        # Return Wx + BA(Wx) + b
        if self.base.bias is not None:
            return wx + lorma_out + self.base.bias

        return wx + lorma_out


# -------------------------------------------------------------------------------------
# Freeze c-6, wrap the chosen linears, check identity at init
# -------------------------------------------------------------------------------------
lm.eval()
probe = identity_probe()
with torch.no_grad():
    ref = features(probe).float()

for p in lm.parameters():
    p.requires_grad = False
n_wrapped = wrap_linears(lm, LORMA_TARGETS, lambda lin: LORMALinear(lin, LORMA_RANK).to(device))

with torch.no_grad():
    diff = (features(probe).float() - ref).abs().max().item()
print(f"wrapped {n_wrapped} linears | LORMA identity-at-init max |diff| = {diff:.2e} (should be ~0)")
assert diff < 1e-3, "LORMA layer is not the identity at init"


# -------------------------------------------------------------------------------------
# [N] Head — YOURS: (B, T, 560) token vectors of B windows -> (B, 2) logits.
#     It has to turn a window's tokens into one vector first (pooling), then classify.
# -------------------------------------------------------------------------------------
class CJPEHead(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()

        self.norm = nn.LayerNorm(in_dim)
        self.fc1 = nn.Linear(in_dim, 256)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(256, n_classes)

    def forward(self, h):
        # h: (B, T, 560)
        # Mean-pool across tokens -> (B, 560)
        h = h.mean(dim=1)

        h = self.norm(h)
        h = self.fc1(h)
        h = self.act(h)
        h = self.dropout(h)
        logits = self.fc2(h)

        # logits: (B, 2)
        return logits

head = CJPEHead(FEAT_DIM, N_CLASSES)
model = CJPEModel(lm, head).to(device)
count_params(model)


# -------------------------------------------------------------------------------------
# Data, training, scoring (plumbing from E0)
# -------------------------------------------------------------------------------------
train_ex = build_examples(train, **BUILD)
val_ex = build_examples(dev, **BUILD)

log, best_res = train_model(
    model, train_ex, val_ex, dev,
    loss_fn=lambda logits, b: ce_loss(logits, b["labels"], CLASS_WEIGHTS),
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
    lr_lambda=LR_LAMBDA, save_path=SAVE, agg=AGG, threshold=THRESHOLD)
train_mem = torch.cuda.max_memory_allocated() / 2**30

# Evidence for the report (section 10.5) with the best weights
model.load_state_dict(torch.load(SAVE), strict=False)
dev_probs = predict_probs(model, val_ex, BATCH_SIZE, AGG)
dev_pred = {k: int(p >= THRESHOLD) for k, p in dev_probs.items()}
evaluate(dev_pred, dev)
confusion(dev_pred, dev)
error_cases(dev_pred, dev_probs, dev)
print(f"dev inference time {time_inference(model, val_ex, BATCH_SIZE, AGG):.1f} s | peak training memory {train_mem:.2f} GB")
json.dump({"run": RUN_NAME, "build": BUILD, "log": log, "dev_probs": dev_probs}, open(f"{OUT_DIR}/dev_log.json", "w"))

# Predict test with the best weights and package -> /kaggle/working/<RUN_NAME>/submission.zip
write_submission(model, BUILD, BATCH_SIZE, AGG, THRESHOLD, best_path=SAVE, out_dir=OUT_DIR)


  class 0: train n=18920  weight=0.914
  class 1: train n=13385  weight=1.086
wrapped 32 linears | LORMA identity-at-init max |diff| = 9.54e-07 (should be ~0)
trainable 12.305M / total 38.36M (32.07%)


windowing:   0%|          | 0/32305 [00:00<?, ?it/s]

32305 cases -> 253583 windows | windows per case: mean 7.85 max 1056


windowing:   0%|          | 0/994 [00:00<?, ?it/s]

994 cases -> 10511 windows | windows per case: mean 10.57 max 138


epoch 1/10:   0%|          | 0/16385 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Cell E3 — KD (teacher = E2 LORMA)

In [ ]:
# =====================================================================================
# CELL E3 — Knowledge distillation on c-6 for CJPE, teacher = your LORMA model from Cell E2
#
#   TEACHER = the fine-tuned model from Cell E2 (`model`: 8-layer c-6 + LORMA + your CJPEHead),
#             with its best weights reloaded.
#   STUDENT = a SHALLOWER c-6: the Phase 1 c-6 checkpoint keeping only KEEP_LAYERS encoder
#             layers, LORMA on those layers (your LORMALinear) and a FRESH CJPEHead.
#   Same tokenizer and the same Phase 2 windows, so the teacher gives one logit pair per
#   training window.
#
# Run right after Cell E2 in the same session (uses its BUILD, train_ex, val_ex, CJPEHead,
# LORMALinear, CLASS_WEIGHTS, SAVE, BATCH_SIZE, AGG, THRESHOLD).
# Writes the student's submission to /kaggle/working/c6_kd_student_<n>L/submission.zip.
# Course rule: every value marked None is yours.
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters — all yours. None = not set; the assert stops you.
# -------------------------------------------------------------------------------------
KEEP_LAYERS     = [0, 2, 4, 6]
S_LORMA_RANK    = 200
S_LORMA_TARGETS = ("qkv_proj", "out_proj", "ffn_in", "ffn_out")

TEMPERATURE     = 2.0
KD_WEIGHT       = 0.5

S_LR            = 1e-4
S_EPOCHS        = 10
S_BATCH_SIZE    = 16
S_WEIGHT_DECAY  = 0.01
S_LR_LAMBDA     = None

SEED            = 42
for _n in ["KEEP_LAYERS", "S_LORMA_RANK", "S_LORMA_TARGETS", "TEMPERATURE", "KD_WEIGHT", "S_LR",
           "S_EPOCHS", "S_BATCH_SIZE", "S_WEIGHT_DECAY"]:
    assert globals()[_n] is not None, f"set {_n}"
assert all(0 <= i < cfg.n_layers for i in KEEP_LAYERS) and len(KEEP_LAYERS) < cfg.n_layers, \
    "the student must keep fewer encoder layers than the teacher"
KD_NAME = f"c6_kd_student_{len(KEEP_LAYERS)}L"
KD_DIR = f"/kaggle/working/{KD_NAME}"; os.makedirs(KD_DIR, exist_ok=True)


# -------------------------------------------------------------------------------------
# 1. Teacher: best E2 LORMA weights, scored on dev
# -------------------------------------------------------------------------------------
teacher = model
teacher.load_state_dict(torch.load(SAVE), strict=False)
teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad = False
T_res = evaluate(predict(teacher, val_ex, BATCH_SIZE, AGG, THRESHOLD), dev, show=False)
print(f"TEACHER dev macro-F1 {T_res['f1']:.4f} (P {T_res['p']:.4f}, R {T_res['r']:.4f})")


# -------------------------------------------------------------------------------------
# 2. Teacher logits for every TRAIN window (once): ex["soft"] = (2,) float32 on CPU.
#    to_tensors (E0) then adds b["soft"] (B, 2) to every training batch.
# -------------------------------------------------------------------------------------
@torch.no_grad()
def add_teacher_logits(examples, batch_size):
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="teacher logits"):
        with torch.autocast("cuda", dtype=torch.float16):
            lg = teacher(to_tensors(batch)["ids"]).float().cpu()
        assert torch.isfinite(lg).all(), "non-finite teacher logits"
        for ex, row in zip(batch, lg):
            ex["soft"] = row

add_teacher_logits(train_ex, BATCH_SIZE)
assert all("soft" in ex for ex in train_ex)
_lab = torch.tensor([ex["label"] for ex in train_ex])
_arg = torch.stack([ex["soft"] for ex in train_ex]).argmax(-1)
print(f"teacher accuracy on train windows: {(_arg == _lab).float().mean():.4f}")
teacher.cpu(); torch.cuda.empty_cache()                 # frees GPU memory for the student


# -------------------------------------------------------------------------------------
# 3. Student: fresh Phase 1 c-6, only KEEP_LAYERS encoder layers, LORMA on them
# -------------------------------------------------------------------------------------
_ck = torch.load(f"{ROOT}/cs781_phase1_full/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
lm_s = EncDecLM(EncDecConfig(**_ck["config"])); print(lm_s.load_state_dict(_ck["model"])); del _ck
lm_s.core.decoder_layers = None                       # decoder unused, as in E0
lm_s.core.decoder_ln = None
lm_s.core.encoder_layers = nn.ModuleList([lm_s.core.encoder_layers[i] for i in KEEP_LAYERS])
lm_s.to(device).eval()
for p in lm_s.parameters():
    p.requires_grad = False
n_s = wrap_linears(lm_s, S_LORMA_TARGETS, lambda lin: LORMALinear(lin, S_LORMA_RANK).to(device))
print(f"student: {len(KEEP_LAYERS)} of {cfg.n_layers} encoder layers {list(KEEP_LAYERS)} | LORMA on {n_s} linears")

# Student head: a NEW CJPEHead (untrained), same design as the teacher's
student = CJPEModel(lm_s, CJPEHead(FEAT_DIM, N_CLASSES)).to(device)
count_params(student)


# -------------------------------------------------------------------------------------
# [K] Distillation loss (Hinton et al. 2015, ref [2]): KL between temperature-softened teacher
#     and student distributions (scaled by T^2), plus gold-label cross-entropy.
#     logits (B, 2) student, labels (B,) gold case label, soft (B, 2) teacher logits, one row per window.
#     >>> Swap in your own kd_loss from the earlier phases if it differs. <<<
# -------------------------------------------------------------------------------------
def kd_loss(logits, labels, soft, class_weights=None):
    T = TEMPERATURE
    kd = F.kl_div(F.log_softmax(logits / T, dim=-1),
                  F.softmax(soft / T, dim=-1),
                  reduction="batchmean") * (T * T)
    ce = ce_loss(logits, labels, class_weights)
    return KD_WEIGHT * kd + (1.0 - KD_WEIGHT) * ce


# -------------------------------------------------------------------------------------
# 4. Train the student (same loop as E2, best dev epoch kept)
# -------------------------------------------------------------------------------------
random.seed(SEED); torch.manual_seed(SEED)
S_SAVE = f"{KD_DIR}/best_trainable.pt"

s_log, S_best = train_model(
    student, train_ex, val_ex, dev,
    loss_fn=lambda logits, b: kd_loss(logits, b["labels"], b["soft"], CLASS_WEIGHTS),
    lr=S_LR, epochs=S_EPOCHS, batch_size=S_BATCH_SIZE, weight_decay=S_WEIGHT_DECAY,
    lr_lambda=S_LR_LAMBDA, save_path=S_SAVE, agg=AGG, threshold=THRESHOLD)
s_train_mem = torch.cuda.max_memory_allocated() / 2**30


# -------------------------------------------------------------------------------------
# 5. Student with its best weights: scores, confusion matrix, error cases
# -------------------------------------------------------------------------------------
student.load_state_dict(torch.load(S_SAVE), strict=False)
S_probs = predict_probs(student, val_ex, S_BATCH_SIZE, AGG)
S_pred = {k: int(p >= THRESHOLD) for k, p in S_probs.items()}
S_res = evaluate(S_pred, dev)
confusion(S_pred, dev)
error_cases(S_pred, S_probs, dev)


# -------------------------------------------------------------------------------------
# 6. Teacher vs student (the KD table for the report). Same batch size for both timings.
# -------------------------------------------------------------------------------------
def _mem_infer(m):
    torch.cuda.reset_peak_memory_stats()
    predict_probs(m, val_ex, S_BATCH_SIZE, AGG)
    return torch.cuda.max_memory_allocated() / 2**30

teacher.to(device)
rows = []
for name, m, res in ((f"teacher c-6 {cfg.n_layers}L", teacher, T_res), (f"student c-6 {len(KEEP_LAYERS)}L", student, S_res)):
    rows.append(dict(name=name, f1=res["f1"], p=res["p"], r=res["r"],
                     params=sum(p.numel() for p in m.parameters()),
                     t=time_inference(m, val_ex, S_BATCH_SIZE, AGG), mem=_mem_infer(m)))
print(f"\n{'model':20s} {'F1':>7s} {'P':>7s} {'R':>7s} {'params':>9s} {'infer s':>8s} {'peak GB':>8s}")
for r in rows:
    print(f"{r['name']:20s} {r['f1']:7.4f} {r['p']:7.4f} {r['r']:7.4f} {r['params']/1e6:8.2f}M {r['t']:8.1f} {r['mem']:8.2f}")
T_r, S_r = rows
print(f"student keeps {100*S_r['f1']/max(T_r['f1'], 1e-9):.1f}% of the teacher's dev macro-F1 with "
      f"{100*S_r['params']/T_r['params']:.1f}% of its parameters; peak training memory {s_train_mem:.2f} GB")
json.dump({"run": KD_NAME, "build": BUILD, "log": s_log, "table": rows, "dev_probs": S_probs},
          open(f"{KD_DIR}/dev_log.json", "w"))


# -------------------------------------------------------------------------------------
# 7. Student prediction on test -> /kaggle/working/c6_kd_student_<n>L/submission.zip
# -------------------------------------------------------------------------------------
write_submission(student, BUILD, S_BATCH_SIZE, AGG, THRESHOLD, best_path=S_SAVE, out_dir=KD_DIR)
